In [ ]:
import pathlib

In [ ]:
import downstream
import imageio
import marimo as mo
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import polars as pl
from tqdm.auto import tqdm
from watermark import watermark

from pylib.munge import addend_groups, squeeze_int, union_upsample

In [ ]:
mo.md(
    f"""
```Text
{watermark(
    current_date=True,
    iso8601=True,
    machine=True,
    updated=True,
    python=True,
    iversions=True,
    globals_=globals(),
)}
```
"""
)

```Text
Last updated: 2026-11-21T00:14:17.224601+00:00

Python implementation: CPython
Python version       : 3.10.12
IPython version      : 7.31.1

Compiler    : GCC 11.4.0
OS          : Linux
Release     : 6.8.0-1070-azure
Machine     : x86_64
Processor   : x86_64
CPU cores   : 4
Architecture: 64bit

downstream: 1.25.0
marimo    : 0.23.2
imageio   : 2.37.0
numpy     : 2.1.2
PIL       : 12.2.0
polars    : 1.40.1

```

# 2026-11-20-alleleevoepi-nestedsubgrid-p2p-traitlog

Animates strains of each PE's sampled host (0th population member)
over time, from the trait logger of wse-async-ga workflow
`2026-11-20-alleleevoepi-nestedsubgrid-p2p-traitlog` (genome
`genome_alleleevoepi_2026_11_19`, 2 sites), nested subgrid runs
sweeping mutation rate.

Nested subgrids: `nRowSubgrid` x `nRowSubgrid` subgrids at the corner
PE (row 0, col 0), then L-shaped shells of `factor`^2 - 1 subgrids,
each `factor` times larger, with subgrids overhanging the grid edge
clipped (wse-async-ga runtime config `nestedSubgridSizeFactor`).

For each run, animates (1) the whole grid, sampling every 8th PE along
each axis, and (2) every full (not clipped) subgrid, each sampled at the
same `nRowSubgrid` x `nRowSubgrid` resolution, so all subgrids show at
the same scale, with subgrid sizes in rows and replicate subgrids in
columns.

Colors: black, uninfected; white, wildtype; blue, site 0 mutant; red,
site 1 mutant; purple, double mutant.

In [ ]:
nbname = pathlib.Path(__file__).stem
outdir = pathlib.Path("outdata") / nbname
outdir.mkdir(parents=True, exist_ok=True)
outdir

PosixPath('outdata/2026-11-20-alleleevoepi-nestedsubgrid-p2p-traitlog')

## Get Data

In [ ]:
# trait logger records, one OSF file per run (runtime config)
osf_ids = ["b4cf5"]

# wse-async-ga runtime config nestedSubgridSizeFactor of these runs,
# which trait logger output doesn't record
nest_factor = 2

# sample every nth PE along each axis for whole-grid animation
grid_stride = 8

config_columns = [
    "dispatchSlug",
    "globalSeed",
    "nRow",
    "nCol",
    "nRowSubgrid",
    "nColSubgrid",
    "tilePopSize",
    "tournSize",
    "nCycle",
    "p2pSendProbability",
    "evoepi_mutation_rate",
    "evoepi_waning_rate",
    "evoepi_seed_pe_stride",
    "dstream_storage_multiplex",
]

## Unpack Data

In [ ]:
def nested_subgrid_size(
    row: np.ndarray, col: np.ndarray, base: int, factor: int
) -> np.ndarray:
    """Size of each PE's nested subgrid."""
    shell = np.maximum(row, col) // base  # base-sized steps from corner
    size = np.full_like(shell, base)
    while (shell >= factor).any():
        grow = shell >= factor
        size[grow] *= factor
        shell[grow] //= factor
    return size

def sample_tiles(
    n_row: int, n_col: int, base: int, factor: int, grid_stride: int
) -> pl.DataFrame:
    """PEs to unpack: whole-grid sample, and each full subgrid sampled
    at base x base resolution (subgrid_i, subgrid_j)."""
    rows, cols = np.indices((n_row, n_col))
    size = nested_subgrid_size(rows, cols, base, factor)
    top, left = rows - rows % size, cols - cols % size
    is_full = (top + size <= n_row) & (left + size <= n_col)
    stride = size // base
    is_subgrid_sample = (
        is_full & ((rows - top) % stride == 0) & ((cols - left) % stride == 0)
    )
    is_grid_sample = (rows % grid_stride == 0) & (cols % grid_stride == 0)
    keep = is_subgrid_sample | is_grid_sample
    return pl.DataFrame(
        {
            "row": rows[keep],
            "col": cols[keep],
            "is_grid_sample": is_grid_sample[keep],
            "is_subgrid_sample": is_subgrid_sample[keep],
            "subgrid_size": size[keep],
            "subgrid_top": top[keep],
            "subgrid_left": left[keep],
            "subgrid_i": ((rows - top) // stride)[keep],
            "subgrid_j": ((cols - left) // stride)[keep],
        },
    ).cast({"row": pl.UInt16, "col": pl.UInt16})

In [ ]:
def unpack_strain_timeseries(df: pl.DataFrame) -> pl.DataFrame:
    # unpack dstream data items from logger hex strings, demultiplexing
    # infected, site 0 allele, and site 1 allele records
    dfu = downstream.dataframe.unpack_data_packed(df)
    df = downstream.dataframe.explode_lookup_unpacked(
        dfu.with_columns(
            # fast forward loggers to at least dstream_S entries
            # (i.e., buffer filled)
            # these will be semantically-correct zero values
            dstream_T=pl.max_horizontal(
                pl.col("dstream_T"), pl.col("dstream_S")
            ),
        ),
        value_type="uint8",
    )

    # join metadata onto unpacked data
    df = df.join(
        dfu.select("dstream_data_id", "col", "row", "tile").unique(
            "dstream_data_id",
        ),
        on="dstream_data_id",
    )

    # apply dilation to dstream_Tbar, then one column per record
    df = (
        df.with_columns(
            dstream_Tbar=pl.col("dstream_Tbar")
            * pl.col("dstream_T_dilation"),
        )
        .pivot(
            on="dstream_storage_multiplex_id",
            index=["tile", "row", "col", "dstream_Tbar"],
            values="dstream_value",
        )
        .rename({"0": "infected", "1": "site0", "2": "site1"})
    )

    # addend time series groups with final uninfected entry, after
    # logger's last seen time (i.e., last time any record was set)
    df = addend_groups(
        df,
        group_by="tile",
        aggs=(
            pl.col("col").unique().pipe(squeeze_int),
            pl.col("row").unique().pipe(squeeze_int),
            (pl.col("dstream_Tbar") + 1).max().alias("dstream_Tbar"),
            pl.lit(0).alias("infected"),
            pl.lit(0).alias("site0"),
            pl.lit(0).alias("site1"),
        ),
        inner_only=True,
    )

    # upsample to fill in dstream_Tbar dropped by some loggers
    df = union_upsample(
        df,
        upsample="dstream_Tbar",
        group_by="tile",
        fill_null_ops=[
            (pl.all(), {"strategy": "forward"}),
            (pl.all(), {"strategy": "backward"}),
        ],
    )
    assert len(df["dstream_Tbar"].value_counts()["count"].unique()) == 1

    # 0: uninfected, 1: wildtype, 2: site 0 mutant, 3: site 1 mutant,
    # 4: double mutant
    return df.with_columns(
        strain=pl.col("infected")
        * (1 + pl.col("site0") + 2 * pl.col("site1")),
    )

In [ ]:
# read and unpack one run at a time, keeping only sampled PEs' strains
_configs, dfs = [], {}
for _osf_id in tqdm(osf_ids):
    _df = pl.read_parquet(
        f"https://osf.io/{_osf_id}/download",
    ).with_row_index("dstream_data_id")
    _config = _df.select(config_columns).unique()
    assert len(_config) == 1
    _configs.append(_config.with_columns(osf_id=pl.lit(_osf_id)))
    _samples = sample_tiles(
        _config["nRow"].item(),
        _config["nCol"].item(),
        _config["nRowSubgrid"].item(),
        nest_factor,
        grid_stride,
    )
    _df = _df.join(_samples, on=["row", "col"])
    dfs[_osf_id] = unpack_strain_timeseries(_df).join(
        _samples, on=["row", "col"]
    )
    del _df

configs = pl.concat(_configs)
assert len(configs) == len(osf_ids)
configs

  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/downstream/dataframe/_impl/_check_downstream_version.py:34: UserWarning: Dataframe downstream_version column not provided
  warnings.warn(
100%|██████████| 1/1 [01:31<00:00, 91.53s/it]


dispatchSlug,globalSeed,nRow,nCol,nRowSubgrid,nColSubgrid,tilePopSize,tournSize,nCycle,p2pSendProbability,evoepi_mutation_rate,evoepi_waning_rate,evoepi_seed_pe_stride,dstream_storage_multiplex,osf_id
cat,u32,u16,u16,u16,u16,u16,f32,u32,f32,f32,f32,u16,u8,str
"""mutlog2-16-seed1124""",1124,1170,755,9,9,6500,2.0,50000,0.02,0.000015,0.002,9,3,"""b4cf5"""


In [ ]:
pl.concat(
    [
        _df.group_by("strain").len().with_columns(osf_id=pl.lit(_osf_id))
        for _osf_id, _df in dfs.items()
    ],
).pivot(on="strain", index="osf_id", values="len", sort_columns=True)

osf_id,0,1,2,3,4
str,u64,u64,u64,u64,u64
"""b4cf5""",13743706,426943,253995,254498,411596


## Animate Data

In [ ]:
palette = np.array(
    [
        [0, 0, 0],  # uninfected: black
        [255, 255, 255],  # wildtype: white
        [0, 0, 255],  # site 0 mutant: blue
        [255, 0, 0],  # site 1 mutant: red
        [160, 32, 240],  # double mutant: purple
    ],
    dtype=np.uint8,
)

In [ ]:
def save_animation(outpath: str, frames: list) -> None:
    imageio.mimsave(f"{outpath}.gif", frames, duration=0.05, loop=0)
    # mp4 lasts 15 seconds, whatever its number of frames
    imageio.mimsave(
        f"{outpath}.mp4",
        frames,
        fps=len(frames) / 15,
        macro_block_size=1,
    )

In [ ]:
def make_run_label(config: dict) -> str:
    return (
        f"mutlog2={round(np.log2(config['evoepi_mutation_rate']))}"
        f"+p2p={config['p2pSendProbability']:.3g}"
        f"+waning={config['evoepi_waning_rate']:.3g}"
        f"+seed={config['globalSeed']}"
    )

def strain_frames(df: pl.DataFrame, index: str) -> np.ndarray:
    """Strains, as array indexed by [time, index value]."""
    df = df.sort("dstream_Tbar", index)
    num_t = df["dstream_Tbar"].n_unique()
    return df["strain"].to_numpy().reshape(num_t, -1).astype(np.uint8)

In [ ]:
# whole grid, every grid_stride-th PE along each axis
for _osf_id, _df in tqdm(dfs.items()):
    _config = configs.filter(pl.col("osf_id") == _osf_id).row(0, named=True)
    _label = make_run_label(_config)
    _df = _df.filter("is_grid_sample").with_columns(
        row_rank=pl.col("row").rank(method="dense").cast(pl.Int32) - 1,
        col_rank=pl.col("col").rank(method="dense").cast(pl.Int32) - 1,
    )
    _num_rows = _df["row_rank"].max() + 1
    _num_cols = _df["col_rank"].max() + 1
    _df = _df.with_columns(cell=pl.col("row_rank") * _num_cols + pl.col("col_rank"))
    assert _df["cell"].n_unique() == _num_rows * _num_cols
    _strains = strain_frames(_df, "cell").reshape(-1, _num_rows, _num_cols)
    _frames = [
        np.kron(palette[_s], np.ones((4, 4, 1), dtype=np.uint8))
        for _s in _strains
    ]
    print(_osf_id, _label, len(_frames), _frames[0].shape)
    save_animation(
        outdir / f"a=strains-grid+{_label}+osf={_osf_id}+ext=", _frames
    )

  0%|          | 0/1 [00:00<?, ?it/s]

b4cf5 mutlog2=-16+p2p=0.02+waning=0.002+seed=1124 1027 (588, 380, 3)


100%|██████████| 1/1 [00:36<00:00, 36.07s/it]


In [ ]:
# every full subgrid, at same resolution; sizes in rows, replicates in cols
_scale, _gap, _label_width = 8, 6, 40
_font = ImageFont.load_default(size=16)
for _osf_id, _df in tqdm(dfs.items()):
    _config = configs.filter(pl.col("osf_id") == _osf_id).row(0, named=True)
    _label = make_run_label(_config)
    _res = _config["nRowSubgrid"]  # sampled PEs per subgrid side
    _df = _df.filter("is_subgrid_sample")
    _subgrids = (
        _df.select("subgrid_size", "subgrid_top", "subgrid_left")
        .unique()
        .sort(pl.all())
        .with_columns(
            replicate=pl.int_range(pl.len()).over("subgrid_size"),
        )
    )
    _sizes = sorted(_subgrids["subgrid_size"].unique())
    _num_reps = _subgrids["replicate"].max() + 1
    _df = _df.join(
        _subgrids, on=["subgrid_size", "subgrid_top", "subgrid_left"]
    ).with_columns(
        cell=(
            (
                pl.col("subgrid_size").replace_strict(
                    _sizes, range(len(_sizes)), return_dtype=pl.Int64
                )
                * _num_reps
                + pl.col("replicate")
            )
            * _res
            + pl.col("subgrid_i")
        )
        * _res
        + pl.col("subgrid_j"),
    )
    assert _df["cell"].n_unique() == len(_subgrids) * _res * _res
    _cells = np.sort(_df["cell"].unique().to_numpy())
    _strains = strain_frames(_df, "cell")

    # layout: label strip, then subgrid cells, gray gaps between
    _cell_px = _res * _scale
    _height = len(_sizes) * (_cell_px + _gap) + _gap
    _width = _label_width + _num_reps * (_cell_px + _gap) + _gap
    _height += _height % 2
    _width += _width % 2
    _background = Image.new("RGB", (_width, _height), (128, 128, 128))
    _draw = ImageDraw.Draw(_background)
    for _k, _size in enumerate(_sizes):  # rotated row labels
        _text = f"{_size} x {_size}"
        _strip = Image.new("RGB", (_cell_px, _label_width), (128, 128, 128))
        _strip_draw = ImageDraw.Draw(_strip)
        _tw = _strip_draw.textlength(_text, font=_font)
        _strip_draw.text(
            ((_cell_px - _tw) / 2, (_label_width - 18) / 2),
            _text,
            font=_font,
            fill=(255, 255, 255),
        )
        _background.paste(
            _strip.rotate(90, expand=True),
            (0, _gap + _k * (_cell_px + _gap)),
        )
    _background = np.asarray(_background)

    # where each subgrid's cells go
    _size_k = _cells // (_res * _res) // _num_reps
    _rep = _cells // (_res * _res) % _num_reps
    _i = _cells // _res % _res
    _j = _cells % _res
    _y = _gap + _size_k * (_cell_px + _gap) + _i * _scale
    _x = _label_width + _gap + _rep * (_cell_px + _gap) + _j * _scale

    _frames = []
    for _s in _strains:
        _frame = _background.copy()
        _colors = palette[_s]
        for _dy in range(_scale):
            for _dx in range(_scale):
                _frame[_y + _dy, _x + _dx] = _colors
        _frames.append(_frame)
    print(_osf_id, _label, len(_frames), _frames[0].shape)
    save_animation(
        outdir / f"a=strains-subgrids+{_label}+osf={_osf_id}+ext=", _frames
    )

  0%|          | 0/1 [00:00<?, ?it/s]

b4cf5 mutlog2=-16+p2p=0.02+waning=0.002+seed=1124 1027 (552, 358, 3)


100%|██████████| 1/1 [00:28<00:00, 28.74s/it]
